# M5 Forecasting: LightGBM pipeline + the winner's recipe, combined

Forecasts 28 days of unit sales for 30,490 Walmart item-stores (d_1942–d_1969) and writes
`submission.csv`. Full write-up, tests and dashboard:
[github.com/Daniel766hi/Simulation-Project](https://github.com/Daniel766hi/Simulation-Project/tree/claude/gifted-tesla-9xrlx2/m5).

**Method.** Two forecasts are averaged 50/50:

1. **This pipeline.** A recursive LightGBM and an origin-anchored multi-horizon LightGBM, one model per
   store, Tweedie loss, per-series scaling with time-decay weights. Their average is aligned to an
   independent store × department aggregate model (top-down alignment, Anderer & Li 2022) and then
   multiplied by per-store bias factors learned on three earlier validation windows.
2. **The M5 winner's recipe** (YeonJun In, 1st place): recursive and non-recursive LightGBM, each per
   store, per store × category and per store × department, averaged. It runs here at the same compute
   as part 1 (255 leaves, 900 trees at learning rate 0.05, last 1,000 days), not at its original size.

**Why the combination.** On three windows that played no part in any choice (d1774–1857), the
winner's recipe scored a mean WRMSSE of 0.626, this pipeline 0.616 and the 50/50 average 0.599.
The two miss the overall level in opposite directions, so averaging cancels much of the error. The
combination was fixed in the repository before any of those results existed.

**Honesty note.** This notebook was written after the competition ended and after the private
leaderboard window was known. Its score is a late submission and says nothing about rank.

**Runtime** on Kaggle's 4-CPU notebooks: about 1.5 h for part 1 and about 6 h more for part 2.
Set `RUN_WINNER = False` for a fast run of part 1 only.

In [ ]:
import glob, os, sys, subprocess, time
from pathlib import Path

RUN_WINNER = True     # add the winner's recipe (about 6 h); False = this pipeline only (about 1.5 h)
QUICK = False         # smoke test: 5 trees per model, finishes in minutes, scores badly

found = glob.glob("/kaggle/input/*/calendar.csv") + glob.glob("/kaggle/input/*/*/calendar.csv")
RAW = Path(os.environ.get("M5_RAW") or (Path(found[0]).parent if found else "."))
WORK = Path(os.environ.get("M5_WORK", "/kaggle/working"))
SRC = WORK / "src"
SRC.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)                                  # the %%writefile cells below write to src/
os.environ["M5_RAW"], os.environ["M5_WORK"] = str(RAW), str(WORK)
ORIGIN = 1941                                   # train through d_1941, forecast d_1942..d_1969
ROUNDS = 5 if QUICK else 800
WIN_ROUNDS = 5 if QUICK else 900
STORE_FACTORS = {"CA_1": 1.001908, "CA_2": 1.048772, "CA_3": 0.971688, "CA_4": 1.026075, "TX_1": 1.018551, "TX_2": 1.013119, "TX_3": 1.019434, "WI_1": 1.01416, "WI_2": 1.052672, "WI_3": 1.040425}
print("data:", RAW, "| work:", WORK)

## Pipeline modules
The repository's own modules, written to `src/`.

In [ ]:
%%writefile src/config.py
"""Paths and competition constants shared by every stage of the pipeline."""
import os
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
# M5_RAW / M5_WORK let the same code run elsewhere, e.g. in a Kaggle notebook.
RAW = Path(os.environ.get("M5_RAW", ROOT / "data" / "raw"))
PROCESSED = Path(os.environ.get("M5_WORK", ROOT / "data")) / "processed"
OUTPUTS = Path(os.environ["M5_WORK"]) / "outputs" if "M5_WORK" in os.environ else ROOT / "outputs"
DASHBOARD_JSON = ROOT.parent / "m5-data.json"

HORIZON = 28
# Day indices (d_1 = 2011-01-29). Public LB = validation, private LB = evaluation.
LAST_TRAIN_VALIDATION = 1913   # train through d_1913, forecast d_1914..d_1941
LAST_TRAIN_EVALUATION = 1941   # train through d_1941, forecast d_1942..d_1969
LAST_DAY = 1969

ID_COLS = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

In [ ]:
%%writefile src/wrmsse.py
"""Weighted Root Mean Squared Scaled Error, the official M5 Accuracy metric.

The score is built over 42,840 series at 12 aggregation levels (total, state, store, category,
department, their crosses, item, item x state, item x store):

    RMSSE_i = sqrt( mean_h (y - yhat)^2 / mean_t (y_t - y_{t-1})^2 )

The denominator is the in-sample one-step naive error, computed from each series' first
non-zero sale so that products not yet on the shelf do not shrink the scale. Each series is
weighted by its dollar sales over the last 28 training days, weights sum to 1 within a level,
and the final score is the plain average of the 12 level scores.

Correctness is checked in `validate_evaluator.py` against the organisers' own weight file and
the published benchmark scores.
"""
import numpy as np
import pandas as pd
from scipy import sparse

from config import HORIZON, ID_COLS, RAW

LEVELS = [
    ("L1", []),
    ("L2", ["state_id"]),
    ("L3", ["store_id"]),
    ("L4", ["cat_id"]),
    ("L5", ["dept_id"]),
    ("L6", ["state_id", "cat_id"]),
    ("L7", ["state_id", "dept_id"]),
    ("L8", ["store_id", "cat_id"]),
    ("L9", ["store_id", "dept_id"]),
    ("L10", ["item_id"]),
    ("L11", ["item_id", "state_id"]),
    ("L12", ["item_id", "store_id"]),
]
LEVEL_NAMES = {
    "L1": "Total", "L2": "State", "L3": "Store", "L4": "Category", "L5": "Department",
    "L6": "State x category", "L7": "State x department", "L8": "Store x category",
    "L9": "Store x department", "L10": "Item", "L11": "Item x state", "L12": "Item x store",
}


def aggregation_matrix(ids: pd.DataFrame):
    """Sparse 0/1 matrix mapping the 30,490 bottom series to all 42,840 hierarchy series."""
    blocks, labels = [], []
    n = len(ids)
    for level, cols in LEVELS:
        if cols:
            key = ids[cols].astype(str).agg("__".join, axis=1)
        else:
            key = pd.Series("Total", index=ids.index)
        codes, uniques = pd.factorize(key, sort=False)
        blocks.append(sparse.csr_matrix(
            (np.ones(n, dtype=np.float32), (codes, np.arange(n))), shape=(len(uniques), n)))
        labels.append(pd.DataFrame({"level": level, "series": uniques}))
    return sparse.vstack(blocks).tocsr(), pd.concat(labels, ignore_index=True)


class WRMSSEEvaluator:
    """Precomputes scales and weights once, then scores any 30,490 x 28 forecast matrix."""

    def __init__(self, sales: pd.DataFrame, prices: pd.DataFrame, calendar: pd.DataFrame,
                 last_train_day: int, actuals: np.ndarray):
        self.ids = sales[ID_COLS].reset_index(drop=True)
        train_cols = [f"d_{d}" for d in range(1, last_train_day + 1)]
        train = sales[train_cols].to_numpy(np.float32)
        self.S, self.labels = aggregation_matrix(self.ids)

        agg_train = self.S @ train                                   # 42,840 x T
        self.scale = self._scales(agg_train)
        self.weights = self._weights(train, prices, calendar, last_train_day)
        self.actuals = np.asarray(actuals, dtype=np.float32)
        self.agg_actuals = self.S @ self.actuals
        self.level_idx = {lv: np.flatnonzero(self.labels["level"].to_numpy() == lv)
                          for lv, _ in LEVELS}

    @staticmethod
    def _scales(agg_train):
        # Squared one-step differences, counted only from the first non-zero observation on.
        started = np.cumsum(agg_train != 0, axis=1) > 0
        diff2 = np.diff(agg_train, axis=1) ** 2
        mask = started[:, :-1]                    # a diff counts once its left point is active
        n = mask.sum(axis=1)
        return (diff2 * mask).sum(axis=1) / np.maximum(n, 1)

    def _weights(self, train, prices, calendar, last_train_day):
        days = range(last_train_day - HORIZON + 1, last_train_day + 1)
        cal = calendar.set_index("d")
        weeks = cal.loc[[f"d_{d}" for d in days], "wm_yr_wk"].to_numpy()
        p = prices.pivot_table(index=["item_id", "store_id"], columns="wm_yr_wk",
                               values="sell_price")
        key = pd.MultiIndex.from_frame(self.ids[["item_id", "store_id"]])
        price_mat = p.reindex(key)[weeks].to_numpy(np.float32)
        dollars = np.nan_to_num(train[:, -HORIZON:] * price_mat)
        agg = self.S @ dollars.sum(axis=1)
        w = np.empty_like(agg)
        for level, _ in LEVELS:
            idx = np.flatnonzero(self.labels["level"].to_numpy() == level)
            w[idx] = agg[idx] / agg[idx].sum()
        return w

    def rmsse(self, forecast: np.ndarray) -> np.ndarray:
        agg_fc = self.S @ np.asarray(forecast, dtype=np.float32)
        mse = ((self.agg_actuals - agg_fc) ** 2).mean(axis=1)
        # A series with no sales history yet (not launched at an early origin) has scale 0 and
        # weight 0; give it zero error instead of 0 * inf = NaN.
        with np.errstate(divide="ignore", invalid="ignore"):
            return np.where(self.scale > 0, np.sqrt(mse / np.where(self.scale > 0, self.scale, 1)), 0.0)

    def score(self, forecast: np.ndarray):
        """Return (overall WRMSSE, {level: score})."""
        r = self.rmsse(forecast)
        per_level = {lv: float((self.weights[i] * r[i]).sum()) for lv, i in self.level_idx.items()}
        return float(np.mean(list(per_level.values()))), per_level


def _calendar(raw):
    calendar = pd.read_csv(raw / "calendar.csv")
    if "d" not in calendar:              # the redistributed calendar drops Kaggle's d column
        calendar.insert(0, "d", [f"d_{i}" for i in range(1, len(calendar) + 1)])
    return calendar


def load_raw():
    sales = pd.read_csv(RAW / "sales_train_evaluation.csv")
    if not (RAW / "sales_test_evaluation.csv").exists():
        # Kaggle ships sales only up to d_1941; the last 28 days are what is being forecast.
        future = pd.DataFrame(np.nan, index=sales.index,
                              columns=[f"d_{d}" for d in range(1942, 1970)])
        return pd.concat([sales, future], axis=1), _calendar(RAW), pd.read_csv(RAW / "sell_prices.csv")
    test = pd.read_csv(RAW / "sales_test_evaluation.csv")
    calendar = _calendar(RAW)
    prices = pd.read_csv(RAW / "sell_prices.csv")
    assert (sales["item_id"].values == test["item_id"].values).all()
    assert (sales["store_id"].values == test["store_id"].values).all()
    full = pd.concat([sales, test.drop(columns=ID_COLS)], axis=1)
    return full, calendar, prices


def build_evaluator(full, calendar, prices, last_train_day):
    """Evaluator for forecasting days last_train_day+1 .. +28 using actuals held in `full`."""
    act_cols = [f"d_{d}" for d in range(last_train_day + 1, last_train_day + HORIZON + 1)]
    return WRMSSEEvaluator(full, prices, calendar, last_train_day, full[act_cols].to_numpy())

In [ ]:
%%writefile src/prepare_data.py
"""Turn the wide M5 files into one long, compact parquet grid per store.

Each row is one item-store-day with its unit sales and every feature that does not depend on
past sales (those are added at training time, because their construction differs between the
direct and recursive models). Rows before an item's first price week are dropped: the product
was not on the shelf, so its zeros are not demand information.
"""
import numpy as np
import pandas as pd

from config import ID_COLS, LAST_DAY, PROCESSED
from wrmsse import load_raw

CAT_COLS = ["item_id", "dept_id", "cat_id", "store_id", "state_id",
            "event_name_1", "event_type_1", "event_name_2", "event_type_2"]


def calendar_features(calendar: pd.DataFrame) -> pd.DataFrame:
    cal = calendar.copy()
    cal["d"] = cal["d"].str[2:].astype(np.int16)
    date = pd.to_datetime(cal["date"])
    cal["tm_dom"] = date.dt.day.astype(np.int8)
    cal["tm_woy"] = date.dt.isocalendar().week.astype(np.int8)
    cal["tm_month"] = date.dt.month.astype(np.int8)
    cal["tm_year"] = (date.dt.year - date.dt.year.min()).astype(np.int8)
    cal["tm_dow"] = date.dt.dayofweek.astype(np.int8)
    cal["tm_weekend"] = (cal["tm_dow"] >= 5).astype(np.int8)
    # Distance to the next / since the last calendar event, so the model can learn the
    # run-up to Christmas, Thanksgiving, Super Bowl etc. rather than only the day itself.
    has_event = cal["event_name_1"].notna().to_numpy()
    idx = np.arange(len(cal))
    ev_idx = idx[has_event]
    nxt = np.searchsorted(ev_idx, idx)
    prv = nxt - 1
    to_next = np.where(nxt < len(ev_idx), ev_idx[np.minimum(nxt, len(ev_idx) - 1)] - idx, 99)
    since = np.where(prv >= 0, idx - ev_idx[np.maximum(prv, 0)], 99)
    cal["days_to_event"] = np.minimum(to_next, 30).astype(np.int8)
    cal["days_since_event"] = np.minimum(since, 30).astype(np.int8)
    keep = ["d", "wm_yr_wk", "event_name_1", "event_type_1", "event_name_2", "event_type_2",
            "snap_CA", "snap_TX", "snap_WI", "tm_dom", "tm_woy", "tm_month", "tm_year",
            "tm_dow", "tm_weekend", "days_to_event", "days_since_event"]
    return cal[keep]


def price_features(prices: pd.DataFrame, calendar: pd.DataFrame) -> pd.DataFrame:
    p = prices.copy()
    g = p.groupby(["store_id", "item_id"])["sell_price"]
    p["price_max"] = g.transform("max")
    p["price_min"] = g.transform("min")
    p["price_std"] = g.transform("std")
    p["price_mean"] = g.transform("mean")
    p["price_norm"] = p["sell_price"] / p["price_max"]          # 1.0 = regular shelf price
    p["price_nunique"] = g.transform("nunique")
    p["item_nunique"] = p.groupby(["store_id", "sell_price"])["item_id"].transform("nunique")
    # Price momentum: relative to last week (a cut shows up as < 1), to the month and year mean.
    p = p.merge(calendar[["wm_yr_wk", "tm_month", "tm_year"]].drop_duplicates("wm_yr_wk"),
                on="wm_yr_wk", how="left")
    p["price_momentum"] = p["sell_price"] / g.shift(1)
    p["price_momentum_m"] = p["sell_price"] / p.groupby(
        ["store_id", "item_id", "tm_year", "tm_month"])["sell_price"].transform("mean")
    p["price_momentum_y"] = p["sell_price"] / p.groupby(
        ["store_id", "item_id", "tm_year"])["sell_price"].transform("mean")
    # Discount depth relative to the item's rolling 4-week max: a cleaner "promotion" signal.
    p["price_disc_4w"] = p["sell_price"] / g.transform(lambda s: s.rolling(4, 1).max())
    p["release_week"] = p.groupby(["store_id", "item_id"])["wm_yr_wk"].transform("min")
    p = p.drop(columns=["tm_month", "tm_year"])
    for c in p.select_dtypes("float64"):
        p[c] = p[c].astype(np.float32)
    return p


def main():
    PROCESSED.mkdir(parents=True, exist_ok=True)
    full, calendar, prices = load_raw()
    cal = calendar_features(calendar)
    pf = price_features(prices, cal)

    # Consistent category codes across stores so one encoder serves every model.
    enc = {}
    for c in ["item_id", "dept_id", "cat_id", "store_id", "state_id"]:
        enc[c] = {v: i for i, v in enumerate(sorted(full[c].unique()))}
    for c in ["event_name_1", "event_type_1", "event_name_2", "event_type_2"]:
        vals = sorted(cal[c].dropna().unique())
        enc[c] = {v: i + 1 for i, v in enumerate(vals)}           # 0 = no event
        cal[c] = cal[c].map(enc[c]).fillna(0).astype(np.int8)
    pd.to_pickle(enc, PROCESSED / "encoders.pkl")

    day_cols = [f"d_{d}" for d in range(1, LAST_DAY + 1)]
    for store, block in full.groupby("store_id", sort=True):
        state = store.split("_")[0]
        long = block.melt(id_vars=ID_COLS, value_vars=day_cols, var_name="d", value_name="sales")
        long["d"] = long["d"].str[2:].astype(np.int16)
        long["sales"] = long["sales"].astype(np.float32)
        long = long.merge(cal, on="d", how="left")
        long["snap"] = long[f"snap_{state}"].astype(np.int8)
        long = long.drop(columns=["snap_CA", "snap_TX", "snap_WI"])
        long = long.merge(pf, on=["store_id", "item_id", "wm_yr_wk"], how="left")
        long = long[long["wm_yr_wk"] >= long["release_week"]]     # also drops NaN release
        long["weeks_on_sale"] = ((long["wm_yr_wk"] // 100 - long["release_week"] // 100) * 52
                                 + long["wm_yr_wk"] % 100 - long["release_week"] % 100
                                 ).astype(np.int16)
        for c in ["item_id", "dept_id", "cat_id", "store_id", "state_id"]:
            long[c] = long[c].map(enc[c]).astype(np.int16)
        long = long.drop(columns=["release_week"]).sort_values(["item_id", "d"])
        long = long.reset_index(drop=True)
        long.to_parquet(PROCESSED / f"grid_{store}.parquet", index=False)
        print(store, f"{len(long):,} rows", f"{long.memory_usage().sum() / 1e6:.0f} MB")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/features.py
"""Sales-history features, computed from an items x days matrix so training and inference share
exactly one code path.

Two feature sets:

* **direct**   - every lag is at least 28 days old, so one model forecasts all 28 days at once
                 with no feedback of its own predictions (robust, no error accumulation).
* **recursive** - short lags (1-14 days) and recent rolling means; forecasting walks forward
                 one day at a time and feeds each day's prediction into the next day's features.

Pre-release days and every day after the training cut-off are NaN in the matrix, so no
feature can ever see a value from the forecast window.
"""
import numpy as np
import pandas as pd

from config import PROCESSED

STATIC = ["item_id", "dept_id", "cat_id", "event_name_1", "event_type_1", "event_name_2",
          "event_type_2", "tm_dom", "tm_woy", "tm_month", "tm_year", "tm_dow", "tm_weekend",
          "days_to_event", "days_since_event", "snap", "sell_price", "price_max", "price_min",
          "price_std", "price_mean", "price_norm", "price_nunique", "item_nunique",
          "price_momentum", "price_momentum_m", "price_momentum_y", "price_disc_4w",
          "weeks_on_sale"]
CATEGORICAL = ["item_id", "dept_id", "cat_id", "event_name_1", "event_type_1",
               "event_name_2", "event_type_2"]

SPECS = {
    "direct": {
        "lags": list(range(28, 43)),
        "rolls": [(28, w) for w in (7, 14, 30, 60, 180)],
        "stds": [(28, w) for w in (7, 30)],
        "same_dow": 28,
    },
    "recursive": {
        "lags": list(range(1, 15)),
        "rolls": [(s, w) for s in (1, 7, 14) for w in (7, 14, 30, 60)],
        "stds": [(1, 7), (1, 30)],
        "same_dow": 7,
    },
}


def load_grid(store: str) -> pd.DataFrame:
    return pd.read_parquet(PROCESSED / f"grid_{store}.parquet")


def to_wide(grid: pd.DataFrame, last_known_day: int, n_days: int = 1969):
    """items x days sales matrix (column j = day j+1), NaN when unknown or not yet on sale."""
    items = np.sort(grid["item_id"].unique())
    row = np.searchsorted(items, grid["item_id"].to_numpy())
    wide = np.full((len(items), n_days), np.nan, dtype=np.float32)
    wide[row, grid["d"].to_numpy() - 1] = grid["sales"].to_numpy()
    wide[:, last_known_day:] = np.nan
    return items, wide


def _rolling(wide, shift, window, func):
    """Rolling nan-aware mean/std of the `window` days ending `shift` days before each column."""
    v = np.nan_to_num(wide)
    c = (~np.isnan(wide)).astype(np.float32)
    pad = np.zeros((wide.shape[0], 1), dtype=np.float64)
    cs = np.concatenate([pad, np.cumsum(v, axis=1, dtype=np.float64)], axis=1)
    cc = np.concatenate([pad, np.cumsum(c, axis=1, dtype=np.float64)], axis=1)
    n = wide.shape[1]
    end = np.arange(n) - shift + 1                    # exclusive end index into cs
    start = end - window
    end_c = np.clip(end, 0, n)
    start_c = np.clip(start, 0, n)
    s = cs[:, end_c] - cs[:, start_c]
    k = cc[:, end_c] - cc[:, start_c]
    with np.errstate(invalid="ignore", divide="ignore"):
        mean = np.where(k > 0, s / k, np.nan)
        if func == "mean":
            return mean.astype(np.float32)
        cs2 = np.concatenate([pad, np.cumsum(v.astype(np.float64) ** 2, axis=1)], axis=1)
        s2 = cs2[:, end_c] - cs2[:, start_c]
        var = np.where(k > 1, s2 / k - mean ** 2, np.nan)
        return np.sqrt(np.maximum(var, 0)).astype(np.float32)


def _shifted(wide, lag):
    out = np.full_like(wide, np.nan)
    out[:, lag:] = wide[:, :-lag]
    return out


def history_features(wide, kind, cols=None):
    """Return {name: items x len(cols) array} for the given day columns (0-based)."""
    spec = SPECS[kind]
    cols = np.arange(wide.shape[1]) if cols is None else np.asarray(cols)
    feats = {}
    for lag in spec["lags"]:
        feats[f"lag_{lag}"] = _shifted(wide, lag)[:, cols]
    for s, w in spec["rolls"]:
        feats[f"rmean_{s}_{w}"] = _rolling(wide, s, w, "mean")[:, cols]
    for s, w in spec["stds"]:
        feats[f"rstd_{s}_{w}"] = _rolling(wide, s, w, "std")[:, cols]
    # Mean of the last four same-weekday observations: the weekly shape, at the right age.
    s0 = spec["same_dow"]
    same = np.stack([_shifted(wide, s0 + 7 * k) for k in range(4)])
    with np.errstate(invalid="ignore"):
        feats[f"dow_mean_{s0}"] = np.nanmean(same, axis=0)[:, cols]
    # Share of zero-sale days in the last 28 known days: intermittency of the item.
    zeros = (wide == 0).astype(np.float32)
    zeros[np.isnan(wide)] = np.nan
    feats["zero_share_28"] = _rolling(zeros, spec["lags"][0], 28, "mean")[:, cols]
    return feats


def item_encodings(wide, last_known_day):
    """Per-item mean and std of sales over the known history (a stable level estimate)."""
    known = wide[:, :last_known_day]
    with np.errstate(invalid="ignore"):
        return {"enc_item_mean": np.nanmean(known, axis=1),
                "enc_item_std": np.nanstd(known, axis=1)}


def assemble(grid, items, wide, kind, days, last_known_day, window=None):
    """Long feature frame for the rows of `grid` whose day is in `days`.

    `window` limits history features to that many days before the first requested day. Every
    feature looks back at most 74 days, so a 200-day window gives identical values while making
    the recursive day-by-day walk far cheaper than recomputing over the full history.
    """
    sub = grid[grid["d"].isin(days)].reset_index(drop=True)
    row = np.searchsorted(items, sub["item_id"].to_numpy())
    day_cols = np.asarray(sorted(days)) - 1
    col_pos = np.searchsorted(day_cols, sub["d"].to_numpy() - 1)
    off = max(0, int(day_cols.min()) - window) if window else 0
    feats = history_features(wide[:, off:], kind, day_cols - off)
    out = sub[["item_id", "d", "sales"] + [c for c in STATIC if c != "item_id"]].copy()
    for name, arr in feats.items():
        out[name] = arr[row, col_pos]
    for name, arr in item_encodings(wide, last_known_day).items():
        out[name] = arr[row].astype(np.float32)
    return out


# --------------------------------------------------------------------------------------------
# Origin-anchored multi-horizon features ("mh" model)
#
# The direct design above uses lags of at least 28 days relative to the *target* day, so on day
# 1 of the horizon it ignores the 27 most recent known days: its level information is always a
# month stale, which under-forecasts a growing business. Here every feature is computed at the
# forecast *origin* from all data up to it, and the horizon h (1..28) is a feature, so one model
# serves every horizon with the freshest data and no recursion. This is the single global
# multi-horizon design of the VN2 inventory-challenge winner (2026 report), including its
# per-series dynamic scaling: sales-history features and the target are divided by the item's
# level at the origin, so the model learns shape rather than volume.
# --------------------------------------------------------------------------------------------
MH_WINDOWS = (7, 14, 28, 56, 112, 364)


def _tail_mean(v, w):
    x = v[:, -w:]
    with np.errstate(invalid="ignore"):
        return np.nanmean(x, axis=1)


def origin_features(wide, origin):
    """Per-item statistics of everything known up to and including day `origin` (1-based)."""
    v = wide[:, :origin]
    f = {f"o_mean_{w}": _tail_mean(v, w) for w in MH_WINDOWS}
    with np.errstate(invalid="ignore"):
        f["o_std_28"] = np.nanstd(v[:, -28:], axis=1)
        z = (v[:, -28:] == 0).astype(np.float32)
        z[np.isnan(v[:, -28:])] = np.nan
        f["o_zero_share_28"] = np.nanmean(z, axis=1)
    for k in range(1, 8):
        f[f"o_lag_{k}"] = v[:, -k]
    sold = np.where(np.nan_to_num(v) > 0, np.arange(v.shape[1]), -1).max(axis=1)
    f["o_days_since_sale"] = np.where(sold >= 0, np.minimum(v.shape[1] - 1 - sold, 365), 365)
    f["o_trend_28_112"] = f["o_mean_28"] / (f["o_mean_112"] + 0.05)
    return f


def mh_level(f):
    lvl = np.fmax(np.nan_to_num(f["o_mean_28"]), 0.25 * np.nan_to_num(f["o_mean_112"]))
    return np.fmax(lvl, 0.05)


def assemble_mh(grid, items, wide, origin, horizon=28):
    """Rows for days origin+1 .. origin+horizon, features anchored at `origin`.

    Returns the feature frame (history features already divided by the level) and the level
    of each row, so target / level is the training label and prediction * level the forecast.
    """
    sub = grid[(grid["d"] > origin) & (grid["d"] <= origin + horizon)].reset_index(drop=True)
    row = np.searchsorted(items, sub["item_id"].to_numpy())
    t = sub["d"].to_numpy().astype(int)                  # 1-based target day
    h = t - origin
    f = origin_features(wide, origin)
    lvl = mh_level(f)[row]
    out = sub[["item_id", "d", "sales"] + [c for c in STATIC if c != "item_id"]].copy()
    out["h"] = h.astype(np.int8)
    scaled = [k for k in f if k not in ("o_days_since_sale", "o_zero_share_28", "o_trend_28_112")]
    for k, arr in f.items():
        out[k] = (arr[row] / lvl if k in scaled else arr[row]).astype(np.float32)
    # The four most recent same-weekday observations at or before the origin, and last year.
    first = t - 7 * np.ceil(h / 7).astype(int)          # most recent same weekday <= origin
    same = np.stack([wide[row, first - 7 * k - 1] for k in range(4)])
    with np.errstate(invalid="ignore"):
        out["o_same_dow_4"] = (np.nanmean(same, axis=0) / lvl).astype(np.float32)
        out["o_same_dow_1"] = (same[0] / lvl).astype(np.float32)
    out["o_last_year"] = (wide[row, t - 364 - 1] / lvl).astype(np.float32)
    out["log_level"] = np.log(lvl).astype(np.float32)
    return out, lvl

In [ ]:
%%writefile src/stockouts.py
"""Hidden stock-outs: finding availability gaps in sales data that has no stock-out flag.

M5 records sales, not demand. When a product is off the shelf its sales are zero, and nothing in
the data says so. A run of 7+ consecutive zero-sale days for an item-store that sold at least one
unit a day on average over the previous 8 weeks is very unlikely to be chance (under a Poisson
with mean 1 it has probability e^-7 < 0.1%; for faster sellers far less), so it is flagged as a
probable stock-out. This is the "stockout-aware" view that the 2026 VN2 inventory-challenge
winner built into its features.

Outputs: how common the gaps are, how much revenue they cost (the item's normal daily rate over
the gap, at shelf price), where they concentrate, and whether they bias the final forecast.
"""
import json

import numpy as np
import pandas as pd

from config import HORIZON, LAST_TRAIN_EVALUATION, OUTPUTS, RAW
from wrmsse import load_raw

MIN_RUN = 7          # consecutive zero-sale days
MAX_RUN = 55         # longer gaps may be delistings rather than stock-outs: reported separately
MIN_RATE = 1.0       # average units/day over the lookback for the item to count as a regular seller
LOOKBACK = 56
PERIOD = (LAST_TRAIN_EVALUATION - 364 + 1, LAST_TRAIN_EVALUATION)   # the last year of history


def zero_runs(row):
    """(start, length) of every run of zeros in a 1-D array."""
    z = np.concatenate([[0], (row == 0).astype(np.int8), [0]])
    d = np.diff(z)
    starts, ends = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    return starts, ends - starts


def price_matrix(full, calendar, prices, first, last):
    cal = calendar.iloc[first - 1:last]
    p = prices.pivot_table(index=["item_id", "store_id"], columns="wm_yr_wk", values="sell_price")
    key = pd.MultiIndex.from_frame(full[["item_id", "store_id"]])
    return p.reindex(key)[cal["wm_yr_wk"].to_numpy()].to_numpy(float)


def main():
    full, calendar, prices = load_raw()
    first, last = PERIOD
    cols = [f"d_{d}" for d in range(first - LOOKBACK, last + 1)]
    Y = full[cols].to_numpy(float)
    P = price_matrix(full, calendar, prices, first - LOOKBACK, last)
    events = []
    for i in range(len(Y)):
        starts, lengths = zero_runs(Y[i])
        for s, L in zip(starts, lengths):
            if L < MIN_RUN or s < LOOKBACK:
                continue
            prior = Y[i, s - LOOKBACK:s]
            if np.isnan(P[i, s - LOOKBACK:s]).all():
                continue                                   # not yet on the shelf
            rate = np.nanmean(prior)
            if rate < MIN_RATE or np.isnan(P[i, s]):
                continue                                   # irregular seller, or delisted
            end = min(s + L, Y.shape[1])
            events.append({"row": i, "start_day": first - LOOKBACK + s, "length": int(L),
                           "rate": float(rate), "lost_units": float(rate * (end - s)),
                           "lost_revenue": float(np.nansum(rate * P[i, s:end])),
                           "open": bool(s + L >= Y.shape[1])})
    all_ev = pd.DataFrame(events)
    all_ev["cat"] = full["cat_id"].to_numpy()[all_ev["row"]]
    all_ev["store"] = full["store_id"].to_numpy()[all_ev["row"]]
    long_gaps = all_ev[all_ev["length"] > MAX_RUN]
    ev = all_ev[all_ev["length"] <= MAX_RUN]
    period_cols = [f"d_{d}" for d in range(first, last + 1)]
    revenue = np.nansum(full[period_cols].to_numpy(float) * P[:, LOOKBACK:])
    regular = (full[[f"d_{d}" for d in range(first - LOOKBACK, first)]].to_numpy(float).mean(1)
               >= MIN_RATE)

    # Do stock-outs bias the forecast? At each forecast origin, compare items with a probable
    # stock-out in the 28 days before it against regular sellers without one.
    def bias_at(o, name):
        path = OUTPUTS / f"preds_{name}_o{o}.npy"
        if not path.exists():
            return None
        recent = all_ev[(all_ev["start_day"] + all_ev["length"] > o - HORIZON)
                        & (all_ev["start_day"] <= o)]["row"].unique()
        fc = np.load(path).sum(axis=1)
        act = full[[f"d_{d}" for d in range(o + 1, o + HORIZON + 1)]].to_numpy(float).sum(axis=1)
        reg = full[[f"d_{d}" for d in range(o - LOOKBACK + 1, o + 1)]].to_numpy(float).mean(1) >= MIN_RATE
        gap = np.zeros(len(full), bool)
        gap[recent] = True
        return {"origin": o, "after_recent_stockout": float(fc[gap].sum() / act[gap].sum()),
                "regular_sellers_without": float(fc[reg & ~gap].sum() / act[reg & ~gap].sum()),
                "items_after_recent_stockout": int(gap.sum())}

    by_origin = [b for b in (bias_at(o, "final") or bias_at(o, "bt_final")
                             for o in (1773, 1801, 1829, 1857, 1885, 1913, 1941)) if b]
    bias = next(b for b in by_origin if b["origin"] == LAST_TRAIN_EVALUATION)

    out = {
        "definition": {"min_run_days": MIN_RUN, "max_run_days": MAX_RUN,
                       "min_rate_units_per_day": MIN_RATE,
                       "lookback_days": LOOKBACK, "period": [first, last]},
        "events": int(len(ev)),
        "item_stores_affected": int(ev["row"].nunique()),
        "regular_item_stores": int(regular.sum()),
        "lost_units": float(ev["lost_units"].sum()),
        "lost_revenue": float(ev["lost_revenue"].sum()),
        "period_revenue": float(revenue),
        "median_length": float(ev["length"].median()),
        "length_hist": {str(k): int(v) for k, v in
                        pd.cut(all_ev["length"], [6, 13, 27, 55, 10_000],
                               labels=["7-13", "14-27", "28-55", "56+"]).value_counts().sort_index().items()},
        "long_gaps": {"events": int(len(long_gaps)),
                      "lost_revenue_if_stockouts": float(long_gaps["lost_revenue"].sum())},
        "by_category": ev.groupby("cat")["lost_revenue"].sum().round(0).to_dict(),
        "by_store": ev.groupby("store")["lost_revenue"].sum().round(0).to_dict(),
        "forecast_bias": bias,
        "forecast_bias_by_origin": by_origin,
    }
    print(json.dumps({k: v for k, v in out.items() if k not in ("by_store",)}, indent=1))
    (OUTPUTS / "stockouts.json").write_text(json.dumps(out, indent=2))
    return out


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/train.py
"""Train pooled LightGBM models and forecast the 28 days after a forecast origin.

    python train.py --kind direct    --pool store     --origin 1913   # public-LB window
    python train.py --kind recursive --pool store_cat --origin 1941   # private-LB window

A pool is the slice of data one model learns from: one model per store (10 models), per store
x category (30) or per store x department (70). The M5 winner averaged all three pools in both
a direct and a recursive variant (Makridakis, Spiliotis & Assimakopoulos, IJF 2022): different
pools see different cross-series patterns, so their errors are partly independent and the
average beats each member. The Tweedie objective suits the target: a point mass at zero (most
item-days sell nothing) plus a long right tail on promotion and holiday days.

Predictions go to outputs/preds_<kind>_<pool>_o<origin>.npy in the row order of the official
sales file so they can be scored or blended directly.
"""
import argparse
import json
import shutil
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

from config import HORIZON, OUTPUTS, PROCESSED, RAW
from features import CATEGORICAL, assemble, assemble_mh, load_grid, to_wide

warnings.filterwarnings("ignore", category=RuntimeWarning)

STORES = ["CA_1", "CA_2", "CA_3", "CA_4", "TX_1", "TX_2", "TX_3", "WI_1", "WI_2", "WI_3"]
TRAIN_DAYS = 1000   # how many days of history each store model learns from

PARAMS = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.1,
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 255,
    "min_data_in_leaf": 255,
    "feature_fraction": 0.6,
    "bagging_fraction": 0.7,
    "bagging_freq": 1,
    "lambda_l2": 0.1,
    "max_bin": 127,
    "boost_from_average": False,
    "num_threads": 4,
    "verbose": -1,
    "seed": 42,
}


POOL_COLS = {"store": None, "store_cat": "cat_id", "store_dept": "dept_id"}

# Tree models cannot extrapolate a level they have not seen (Januschowski et al., "Forecasting
# with trees", IJF 2022), and M5 demand grew 14-20% year on year. Two remedies from the VN2
# winning solution (2026 report): per-series dynamic scaling, so the model learns patterns
# relative to each item's recent level instead of absolute volumes, and time-decayed sample
# weights, so recent behaviour counts more. Whole-history item means are dropped because they
# anchor forecasts to old volume levels.
OPTIONS = {"scale": True, "decay_half_life": 365, "drop_enc": True, "drop_item_id": True}
LEVEL_FEATURE = {"direct": ("rmean_28_30", "rmean_28_180"), "recursive": ("rmean_1_30", "rmean_1_60")}


def level_of(X, kind):
    short, long = LEVEL_FEATURE[kind]
    lvl = np.fmax(X[short].to_numpy(), 0.25 * np.nan_to_num(X[long].to_numpy()))
    return np.fmax(np.nan_to_num(lvl), 0.05)


def scale_frame(X, kind, opts):
    """Divide every sales-history feature by the item's recent level; return frame and level."""
    X = X.drop(columns=[c for c in ("enc_item_mean", "enc_item_std") if opts["drop_enc"]
                        and c in X.columns])
    if not opts["scale"]:
        return X, np.ones(len(X))
    lvl = level_of(X, kind)
    hist = [c for c in X.columns if c.startswith(("lag_", "rmean_", "rstd_", "dow_mean_"))]
    X[hist] = X[hist].to_numpy() / lvl[:, None]
    X["log_level"] = np.log(lvl)
    return X, lvl


MH_ORIGINS = 26          # training origins for the multi-horizon model, one every MH_STEP days
MH_STEP = 14


def train_group_mh(grid, label, last_train, rounds, params, log, opts):
    """Origin-anchored multi-horizon model: one pass, freshest data, every horizon at once."""
    items, wide = to_wide(grid, last_train)
    origins = [last_train - 28 - MH_STEP * k for k in range(MH_ORIGINS)]
    parts = []
    for o in origins:
        block, lvl = assemble_mh(grid, items, wide, o)
        block["_lvl"] = lvl
        block["_age"] = last_train - o
        parts.append(block)
    X = pd.concat(parts, ignore_index=True)
    X = X[X["sales"].notna()].reset_index(drop=True)
    if opts.get("mask_stockouts"):
        m = stockout_mask(wide, last_train)
        hit = m[np.searchsorted(items, X["item_id"].to_numpy()), X["d"].to_numpy() - 1]
        X = X[~hit].reset_index(drop=True)
    # item_id as a 3,049-level categorical lets the model memorise each item's past ratio to its
    # level; out of sample that over-shrinks slow movers, so it is dropped by default.
    drop = {"d", "sales", "_lvl", "_age"} | ({"item_id"} if opts.get("drop_item_id") else set())
    feat_cols = [c for c in X.columns if c not in drop]
    lvl = X["_lvl"].to_numpy()
    weight = lvl * (0.5 ** (X["_age"].to_numpy() / opts["decay_half_life"])
                    if opts["decay_half_life"] else 1.0)
    ds = lgb.Dataset(X[feat_cols], X["sales"].to_numpy() / lvl, weight=weight,
                     categorical_feature=[c for c in CATEGORICAL if c in feat_cols],
                     free_raw_data=True)
    t = time.time()
    model = lgb.train(params, ds, num_boost_round=rounds)
    log(f"  {label}: {len(X):,} rows, {len(feat_cols)} features, trained in {time.time() - t:.0f}s")
    del X, ds
    F, lv = assemble_mh(grid, items, wide, last_train)
    pred = np.clip(model.predict(F[feat_cols]) * lv, 0, None)
    imp = pd.Series(model.feature_importance("gain"), index=feat_cols)
    return F[["item_id", "d"]].assign(pred=pred), imp


def stockout_mask(wide, last_train):
    """True on item-days inside a probable stock-out known by last_train: a run of 7+ zero days
    (at most 55 if it has ended) in an item that sold 1+ unit a day over the 56 days before it.
    Same rule as stockouts.py; only history up to last_train is used."""
    from stockouts import LOOKBACK, MAX_RUN, MIN_RATE, MIN_RUN, zero_runs
    mask = np.zeros(wide.shape, dtype=bool)
    hist = wide[:, :last_train]
    for i in range(len(hist)):
        row = np.nan_to_num(hist[i], nan=-1.0)            # not on sale yet: never a zero run
        for s, L in zip(*zero_runs(row)):
            if L < MIN_RUN or s < LOOKBACK or (s + L < last_train and L > MAX_RUN):
                continue
            if np.nanmean(hist[i, s - LOOKBACK:s]) >= MIN_RATE:
                mask[i, s:s + L] = True
    return mask


def train_group(grid, label, kind, last_train, rounds, params, log, opts=None):
    opts = opts or OPTIONS
    if kind == "mh":
        return train_group_mh(grid, label, last_train, rounds, params, log, opts)
    items, wide = to_wide(grid, last_train)
    train_days = range(last_train - TRAIN_DAYS + 1, last_train + 1)
    X = assemble(grid, items, wide, kind, train_days, last_train)
    X = X[X["sales"].notna()].reset_index(drop=True)
    if opts.get("mask_stockouts"):
        # Censored demand: a zero during a stock-out is not a zero demand, so drop those targets.
        m = stockout_mask(wide, last_train)
        hit = m[np.searchsorted(items, X["item_id"].to_numpy()), X["d"].to_numpy() - 1]
        X = X[~hit].reset_index(drop=True)
    X, lvl = scale_frame(X, kind, opts)
    feat_cols = [c for c in X.columns if c not in ("d", "sales")]
    weight = lvl.copy()                               # keep the loss on the unit scale
    if opts["decay_half_life"]:
        weight *= 0.5 ** ((last_train - X["d"].to_numpy()) / opts["decay_half_life"])
    ds = lgb.Dataset(X[feat_cols], X["sales"].to_numpy() / lvl, weight=weight,
                     categorical_feature=CATEGORICAL, free_raw_data=True)
    t = time.time()
    model = lgb.train(params, ds, num_boost_round=rounds)
    log(f"  {label}: {len(X):,} rows, {len(feat_cols)} features, trained in {time.time() - t:.0f}s")
    del X, ds

    fc_days = list(range(last_train + 1, last_train + HORIZON + 1))
    def predict(F):
        Fs, lv = scale_frame(F.copy(), kind, opts)
        return np.clip(model.predict(Fs[feat_cols]) * lv, 0, None)

    if kind == "direct":
        F = assemble(grid, items, wide, kind, fc_days, last_train)
        pred = predict(F)
        out = F[["item_id", "d"]].assign(pred=pred)
    else:
        # Walk forward: each day's prediction becomes history for the next day's features.
        parts = []
        for day in fc_days:
            F = assemble(grid, items, wide, kind, [day], last_train, window=200)
            pred = predict(F)
            rows = np.searchsorted(items, F["item_id"].to_numpy())
            wide[rows, day - 1] = pred
            parts.append(F[["item_id", "d"]].assign(pred=pred))
        out = pd.concat(parts, ignore_index=True)
    imp = pd.Series(model.feature_importance("gain"), index=feat_cols)
    return out, imp


def train_store(store, pool, kind, last_train, rounds, params, log):
    grid = load_grid(store)
    col = POOL_COLS[pool]
    if col is None:
        return train_group(grid, store, kind, last_train, rounds, params, log)
    outs, imps = [], []
    for key, sub in grid.groupby(col):
        out, imp = train_group(sub.reset_index(drop=True), f"{store}/{col}={key}", kind,
                               last_train, rounds, params, log)
        outs.append(out)
        imps.append(imp)
    return pd.concat(outs, ignore_index=True), pd.concat(imps, axis=1).sum(axis=1)


def to_matrix(pred_long, last_train):
    """Reshape {store: long predictions} into the 30,490 x 28 official row order."""
    enc = pd.read_pickle(PROCESSED / "encoders.pkl")
    order = pd.read_csv(RAW / "sales_train_evaluation.csv", usecols=["item_id", "store_id"])
    order["item_code"] = order["item_id"].map(enc["item_id"])
    mat = np.zeros((len(order), HORIZON), dtype=np.float32)
    for store, df in pred_long.items():
        rows = order.index[order["store_id"] == store]
        pos = pd.Series(np.arange(len(rows)), index=order.loc[rows, "item_code"].to_numpy())
        r = rows[pos.loc[df["item_id"].to_numpy()].to_numpy()]
        mat[r, df["d"].to_numpy() - last_train - 1] = df["pred"].to_numpy()
    return mat


def main():
    global TRAIN_DAYS
    ap = argparse.ArgumentParser()
    ap.add_argument("--kind", choices=["direct", "recursive", "mh"], default="direct")
    ap.add_argument("--pool", choices=list(POOL_COLS), default="store")
    ap.add_argument("--origin", type=int, default=1913,
                    help="last training day; 1913 = public LB, 1941 = private LB, 1885 = extra fold")
    ap.add_argument("--rounds", type=int, default=800)
    ap.add_argument("--stores", default="all")
    ap.add_argument("--tag", default="", help="variant name, appended to the model kind")
    ap.add_argument("--train-days", type=int, default=TRAIN_DAYS)
    ap.add_argument("--no-scaling", action="store_true",
                    help="first-version settings: no dynamic scaling, no decay, item means kept")
    ap.add_argument("--mask-stockouts", action="store_true",
                    help="drop training targets that fall inside a probable stock-out")
    ap.add_argument("--params", default="{}", help="JSON overrides for PARAMS")
    args = ap.parse_args()

    last_train = args.origin
    stores = STORES if args.stores == "all" else args.stores.split(",")
    params = {**PARAMS, **json.loads(args.params)}
    name = f"{args.kind}{args.tag}_{args.pool}_o{last_train}"
    TRAIN_DAYS = args.train_days
    if args.no_scaling:
        OPTIONS.update({"scale": False, "decay_half_life": 0, "drop_enc": False})
    if args.mask_stockouts:
        OPTIONS["mask_stockouts"] = True
    OUTPUTS.mkdir(exist_ok=True)
    log_file = open(OUTPUTS / f"log_{name}.txt", "a")
    # Per-store checkpoints, so a run interrupted by a container restart resumes where it stopped.
    ckpt = OUTPUTS / "checkpoints" / name
    ckpt.mkdir(parents=True, exist_ok=True)

    def log(msg):
        print(msg, flush=True)
        log_file.write(msg + "\n")
        log_file.flush()

    log(f"{name}: train <= d_{last_train}, {args.rounds} rounds, params {params}")
    preds, imps = {}, []
    for store in stores:
        p_file, i_file = ckpt / f"{store}_pred.parquet", ckpt / f"{store}_imp.parquet"
        if p_file.exists() and i_file.exists():
            preds[store], imp = pd.read_parquet(p_file), pd.read_parquet(i_file)["imp"]
            log(f"  {store}: resumed from checkpoint")
        else:
            preds[store], imp = train_store(store, args.pool, args.kind, last_train, args.rounds,
                                            params, log)
            preds[store].to_parquet(p_file)
            imp.rename("imp").to_frame().to_parquet(i_file)
        imps.append(imp.rename(store))
    np.save(OUTPUTS / f"preds_{name}.npy", to_matrix(preds, last_train))
    pd.concat(imps, axis=1).to_csv(OUTPUTS / f"importance_{name}.csv")
    log("done")
    shutil.rmtree(ckpt)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/reconcile.py
"""Top-down alignment: correct bottom-level forecasts with an independent aggregate model.

Why: a model trained on item-store rows minimises item-level loss. Small biases that are
invisible on one item (a store drifting up 3%, a department slowing) add up across 3,049
items, and the upper levels of WRMSSE punish them hard. The M5 runner-up forecast the top
levels separately and adjusted the bottom forecasts towards them (Anderer & Li, "Hierarchical
forecasting with a top-down alignment of independent-level forecasts", IJF 2022); the
reconciliation literature (Athanasopoulos, Hyndman, Kourentzes & Panagiotelis, "Forecast
reconciliation: A review", IJF 2024) explains why mixing information from several levels
beats pure bottom-up.

What this does:
1. Forecast the 70 store x department daily totals with a small global LightGBM trained on
   those aggregates only. Aggregates are smooth, so four horizon-bucket models are used
   (days 1-7 may use lags from 7 days back, days 8-14 from 14, ...): no recursion, no leakage.
2. For every store x department and day, scale the item forecasts so they move towards the
   aggregate forecast:  item *= clip(aggregate / sum(items), 0.75, 1.33) ** alpha.
3. Choose alpha on the rolling validation folds, never on the private window.
"""
import json
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd

from config import HORIZON, OUTPUTS, RAW
from wrmsse import aggregation_matrix, load_raw

warnings.filterwarnings("ignore", category=RuntimeWarning)

AGG_LEVEL = "L9"            # store x department
TRAIN_DAYS = 1000
PARAMS = {"objective": "l2", "learning_rate": 0.03, "num_leaves": 31, "min_data_in_leaf": 40,
          "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1,
          "lambda_l2": 1.0, "verbose": -1, "num_threads": 2, "seed": 7}
ROUNDS = 600
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]


def aggregate_series(full):
    ids = full[["item_id", "dept_id", "cat_id", "store_id", "state_id"]]
    S, labels = aggregation_matrix(ids)
    rows = np.flatnonzero(labels["level"].to_numpy() == AGG_LEVEL)
    S9 = S[rows]
    days = [f"d_{d}" for d in range(1, 1970)]
    A = np.asarray(S9 @ full[days].to_numpy(np.float32))
    keys = labels["series"].iloc[rows].str.split("__", expand=True)
    keys.columns = ["store_id", "dept_id"]
    return S9, A, keys.reset_index(drop=True)


def calendar_frame(calendar):
    cal = calendar.copy()
    date = pd.to_datetime(cal["date"])
    ev = cal["event_name_1"].notna().to_numpy()
    idx = np.arange(len(cal))
    ev_idx = idx[ev]
    nxt = np.searchsorted(ev_idx, idx)
    to_next = np.where(nxt < len(ev_idx), ev_idx[np.minimum(nxt, len(ev_idx) - 1)] - idx, 99)
    return pd.DataFrame({
        "dow": date.dt.dayofweek, "dom": date.dt.day, "month": date.dt.month,
        "event": cal["event_type_1"].astype("category").cat.codes,
        "event_name": cal["event_name_1"].astype("category").cat.codes,
        "days_to_event": np.minimum(to_next, 30),
        "snap_CA": cal["snap_CA"], "snap_TX": cal["snap_TX"], "snap_WI": cal["snap_WI"],
    })


def build_rows(A, keys, cal, days, min_lag, origin):
    """Feature rows for every aggregate series on the given days (1-based day numbers)."""
    known = A.copy()
    known[:, origin:] = np.nan                 # nothing after the origin is ever visible
    scale = np.nanmean(known[:, origin - 364:origin], axis=1, keepdims=True)
    Z = known / scale
    frames = []
    for d in days:
        j = d - 1
        f = {"series": np.arange(len(A)), "day": np.full(len(A), d)}
        for lag in list(range(min_lag, min_lag + 7)) + [min_lag + 7, min_lag + 14, 364]:
            f[f"lag_{lag}"] = Z[:, j - lag] if j - lag >= 0 else np.nan
        for w in (7, 28, 91):
            lo = j - min_lag - w + 1
            f[f"rmean_{w}"] = np.nanmean(Z[:, max(lo, 0):j - min_lag + 1], axis=1)
        f["dow_mean"] = np.nanmean(np.stack([Z[:, j - min_lag - 7 * k] for k in range(4)]), axis=0)
        for c in cal.columns:
            if not c.startswith("snap"):
                f[c] = np.full(len(A), cal[c].iloc[j])
        f["snap"] = np.array([cal[f"snap_{s[:2]}"].iloc[j] for s in keys["store_id"]])
        f["y"] = A[:, j] / scale[:, 0]
        frames.append(pd.DataFrame(f))
    df = pd.concat(frames, ignore_index=True)
    df["store"] = pd.Categorical(keys["store_id"].to_numpy()[df["series"]]).codes
    df["dept"] = pd.Categorical(keys["dept_id"].to_numpy()[df["series"]]).codes
    return df, scale[:, 0]


def forecast_aggregates(A, keys, cal, origin):
    """70 x 28 forecast of store x department totals for days origin+1 .. origin+28."""
    out = np.zeros((len(A), HORIZON))
    for b in range(4):
        min_lag = 7 * (b + 1)
        tr_days = range(origin - TRAIN_DAYS + 1, origin + 1)
        tr, _ = build_rows(A, keys, cal, tr_days, min_lag, origin)
        feats = [c for c in tr.columns if c not in ("y", "series", "day")]
        model = lgb.train(PARAMS, lgb.Dataset(tr[feats], tr["y"],
                                              categorical_feature=["store", "dept"]), ROUNDS)
        fc_days = list(range(origin + 7 * b + 1, origin + 7 * b + 8))
        te, scale = build_rows(A, keys, cal, fc_days, min_lag, origin)
        pred = model.predict(te[feats]) * scale[te["series"]]
        out[te["series"].to_numpy(), te["day"].to_numpy() - origin - 1] = np.maximum(pred, 0)
    return out


def align(bottom, S9, agg_fc, alpha, lo=0.75, hi=1.33):
    sums = np.asarray(S9 @ bottom)
    factor = np.clip(np.where(sums > 0, agg_fc / np.maximum(sums, 1e-9), 1.0), lo, hi) ** alpha
    group = np.asarray(S9.argmax(axis=0)).ravel()      # each item-store belongs to one group
    return bottom * factor[group]


def main(base_name="ensemble", origins_tune=(1885, 1913), origin_final=1941):
    from score import evaluator
    full, calendar, prices = load_raw()
    cal = calendar_frame(calendar)
    S9, A, keys = aggregate_series(full)
    agg = {o: forecast_aggregates(A, keys, cal, o) for o in (*origins_tune, origin_final)}
    for o, fc in agg.items():
        np.save(OUTPUTS / f"agg_L9_o{o}.npy", fc)

    result = {"level": AGG_LEVEL, "alphas": ALPHAS, "folds": {}}
    for o in origins_tune:
        base = np.load(OUTPUTS / f"preds_{base_name}_o{o}.npy")
        act = A[:, o:o + HORIZON]
        bu = np.asarray(S9 @ base)
        result["folds"][o] = {
            "agg_model_wape": float(np.abs(agg[o] - act).sum() / act.sum()),
            "bottom_up_wape": float(np.abs(bu - act).sum() / act.sum()),
            "wrmsse": {str(a): evaluator(o).score(align(base, S9, agg[o], a))[0] for a in ALPHAS},
        }
        print(o, json.dumps(result["folds"][o], indent=1))
    mean = {a: np.mean([result["folds"][o]["wrmsse"][str(a)] for o in origins_tune]) for a in ALPHAS}
    best = min(mean, key=mean.get)
    result["mean_wrmsse_by_alpha"] = {str(a): float(v) for a, v in mean.items()}
    result["alpha"] = best
    print("chosen alpha", best, mean)
    for o in (*origins_tune, origin_final):
        base = np.load(OUTPUTS / f"preds_{base_name}_o{o}.npy")
        np.save(OUTPUTS / f"preds_{base_name}_aligned_o{o}.npy",
                align(base, S9, agg[o], best).astype(np.float32))
    (OUTPUTS / "reconcile.json").write_text(json.dumps(result, indent=2))
    return result


if __name__ == "__main__":
    main()

## Run

In [ ]:
def run(*args):
    """Run one pipeline script from src/, streaming its log."""
    t = time.time()
    print(">>", " ".join(args), flush=True)
    p = subprocess.run([sys.executable, *args], cwd=SRC, env=os.environ.copy(),
                       capture_output=True, text=True)
    print(p.stdout[-3000:], p.stderr[-3000:] if p.returncode else "")
    if p.returncode:
        raise RuntimeError(f"{args[0]} failed")
    print(f"   done in {(time.time() - t) / 60:.1f} min", flush=True)

In [ ]:
# One long parquet grid per store: sales, calendar and price features.
run("prepare_data.py")

In [ ]:
# Part 1: the two weighted members of this pipeline.
V2 = '{"seed":7,"num_leaves":127,"feature_fraction":0.5,"bagging_seed":7,"feature_fraction_seed":7}'
run("train.py", "--kind", "recursive", "--tag", "2", "--train-days", "730", "--params", V2,
    "--pool", "store", "--origin", str(ORIGIN), "--rounds", str(ROUNDS))
run("train.py", "--kind", "mh", "--pool", "store", "--origin", str(ORIGIN), "--rounds", str(ROUNDS))

In [ ]:
# Part 2: the M5 winner's recipe at equal compute (six components).
WIN = '{"min_data_in_leaf":4095,"feature_fraction":0.5,"bagging_fraction":0.5,"bagging_freq":1,"max_bin":100}'
if RUN_WINNER:
    for kind in ("recursive", "direct"):
        for pool in ("store", "store_cat", "store_dept"):
            run("train.py", "--kind", kind, "--tag", "_win", "--no-scaling", "--pool", pool,
                "--origin", str(ORIGIN), "--rounds", str(WIN_ROUNDS), "--params", WIN)

## Combine and write the submission

In [ ]:
# Ensemble -> top-down alignment -> store calibration, then the 50/50 combination.
import numpy as np, pandas as pd
sys.path.insert(0, str(SRC))
import reconcile
from config import OUTPUTS
from wrmsse import load_raw

full, calendar, _ = load_raw()
load = lambda name: np.load(OUTPUTS / f"preds_{name}_o{ORIGIN}.npy")
ensemble = 0.5 * load("recursive2_store") + 0.5 * load("mh_store")
S9, A, keys = reconcile.aggregate_series(full)
agg = reconcile.forecast_aggregates(A, keys, reconcile.calendar_frame(calendar), ORIGIN)
ours = reconcile.align(ensemble, S9, agg, 0.5) * full["store_id"].map(STORE_FACTORS).to_numpy()[:, None]
if RUN_WINNER:
    winner = np.mean([load(f"{k}_win_{p}") for k in ("recursive", "direct")
                      for p in ("store", "store_cat", "store_dept")], axis=0)
    forecast = 0.5 * winner + 0.5 * ours
else:
    forecast = ours
print("forecast", forecast.shape, "total units", round(float(forecast.sum())))

In [ ]:
# submission.csv: evaluation rows = the forecast; validation rows = the known sales of d_1914..d_1941.
F = [f"F{i}" for i in range(1, 29)]
ev = pd.DataFrame(forecast, columns=F)
ev.insert(0, "id", (full["item_id"] + "_" + full["store_id"] + "_evaluation").to_numpy())
va = pd.DataFrame(full[[f"d_{d}" for d in range(1914, 1942)]].to_numpy(), columns=F)
va.insert(0, "id", ev["id"].str.replace("_evaluation", "_validation", regex=False).to_numpy())
sub = pd.concat([va, ev], ignore_index=True)
sample = RAW / "sample_submission.csv"
if sample.exists():
    order = pd.read_csv(sample, usecols=["id"])["id"]
    sub = sub.set_index("id").loc[order].reset_index()
assert len(sub) == 60980 and sub[F].notna().all().all() and (sub[F] >= 0).all().all()
sub.to_csv(WORK / "submission.csv", index=False)
print(sub.shape, "->", WORK / "submission.csv")
sub.head()